In [1]:
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import eigs
import multiprocessing as mp
import os
import time
from numba import njit

# ================== 1. 资源与路径配置 ==================
SAVE_DIR = "riemann_10k_harvest_pure_v2"  # 标注为 V2，代表引入 1/ln^3 修正和高斯溅射
if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)

# 加载权威真值
try:
    TRUE_GAMMAS = np.load("riemann_10k_true.npy")
    print(f"✅ 成功加载权威真值，共 {len(TRUE_GAMMAS)} 个零点。")
except:
    print("❌ 错误：未找到 riemann_10k_true.npy")
    exit()

# ================== 2. 高斯核概率溅射引擎 (Harvest 专供) ==================
@njit(fastmath=True, nogil=True)
def gaussian_harvest_kernel(u_c, k_base, beta, gamma, steps, n_bins, eps):
    # 模拟 Fokker-Planck 扩散过程
    transitions = np.zeros((n_bins, n_bins), dtype=np.float64)
    V = np.zeros(n_bins, dtype=np.float64)
    dx = 2.0 / n_bins
    
    # 初始化：宇宙起始于 x=0.5
    init_bin = int((0.5 + 1.0) / dx)
    V[min(max(init_bin, 0), n_bins-1)] = 1.0 
    
    inv_2eps2 = 1.0 / (2.0 * eps**2)
    radius = int(5.0 * eps / dx) + 1  # 5 sigma 溅射半径
    
    # 物理演化循环
    for n in range(1, steps + 1):
        # 🌟 核心：三阶渐近线修正 (对应您刚跑完的 256 核数据)
        ln_n = np.log(n + 100) # 偏移量保护
        mu = u_c + k_base/(ln_n**2) + beta/(ln_n**2) + gamma/(ln_n**3)
        mu = max(0.1, min(2.0, mu)) # 物理保险丝
        
        V_next = np.zeros(n_bins, dtype=np.float64)
        
        # 对当前存活的概率密度进行溅射
        for i in range(n_bins):
            if V[i] < 1e-12: continue
            
            x = -1.0 + dx * 0.5 + i * dx
            x_next = 1.0 - mu * x * x
            
            j_center = int((x_next + 1.0) / dx)
            j_start = max(0, j_center - radius)
            j_end = min(n_bins - 1, j_center + radius)
            
            # 高斯分配
            w_sum = 0.0
            for j in range(j_start, j_end + 1):
                cj = -1.0 + dx * 0.5 + j * dx
                dist_sq = (cj - x_next) * (cj - x_next)
                w_sum += np.exp(-dist_sq * inv_2eps2)
            
            if w_sum > 1e-18:
                inv_w_sum = 1.0 / w_sum
                for j in range(j_start, j_end + 1):
                    cj = -1.0 + dx * 0.5 + j * dx
                    dist_sq = (cj - x_next) * (cj - x_next)
                    prob = np.exp(-dist_sq * inv_2eps2) * inv_w_sum
                    flow = V[i] * prob
                    V_next[j] += flow
                    transitions[i, j] += flow
        V = V_next
    return transitions

# ================== 3. 狙击手 Worker (Harvest 逻辑) ==================
def harvest_worker(task):
    seg_idx, start_n, end_n, (k0, beta, gamma) = task
    
    U_C = 1.543689012  # 天选临界点
    N_BINS = 8000      # 既然内存够，格子开细点，提高光谱纯度
    STEPS = 500000     # Harvest 阶段不需要 10^10，溅射引擎 50万步足以让转移矩阵稳定
    EPSILON = 0.001    # 物理底噪
    
    try:
        t0 = time.time()
        # 调用溅射引擎
        trans = gaussian_harvest_kernel(U_C, k0, beta, gamma, STEPS, N_BINS, EPSILON)
        
        # 矩阵归一化
        P = sp.csr_matrix(trans)
        row_sums = np.array(P.sum(axis=1)).flatten()
        row_sums[row_sums == 0] = 1.0
        P.data /= row_sums[P.indices]
        
        # 特征值提取：翻倍索要量以应对正能量过滤
        num_targets = end_n - start_n
        calc_k = num_targets * 2 + 100 
        
        vals, _ = eigs(P, k=calc_k, which='LM', tol=1e-5)
        
        # 🌟 核心过滤器：正能量、模长、非实数
        pure_vals = vals[(np.abs(vals) > 0.3) & (vals.imag > 1e-6)]
        phases = np.sort(np.angle(pure_vals))
        
        if len(phases) > 0:
            true_segment = TRUE_GAMMAS[start_n:end_n]
            # 采用 slope 拟合而非简单对齐，以检测最高精度
            from scipy.stats import linregress
            min_len = min(len(phases), len(true_segment))
            slope, intercept, r_val, _, _ = linregress(phases[:min_len], true_segment[:min_len])
            pred = slope * phases[:min_len] + intercept
            avg_err = np.mean(np.abs(pred - true_segment[:min_len]))
            max_err = np.max(np.abs(pred - true_segment[:min_len]))
        else:
            avg_err = 999.9
            max_err = 999.9

        # 保存数据
        filename = os.path.join(SAVE_DIR, f"seg_{seg_idx:03d}_err_{avg_err:.3f}.npy")
        np.save(filename, phases)
        
        return f"Seg {seg_idx:03d} | n={start_n+1}-{end_n} | R2={r_val**2:.6f} | AvgErr={avg_err:.3f} | MaxErr={max_err:.3f} | {time.time()-t0:.1f}s"
    
    except Exception as e:
        return f"Seg {seg_idx} Error: {str(e)}"

# ================== 4. 战役总指挥 ==================
if __name__ == "__main__":
    # 这里填入您 256 核跑出来的拟合参数 (以下为示例值，请根据您的结果修改)
    # k_n = K_BASE + BETA/ln^2 + GAMMA/ln^3
    K_BASE = 4.7000
    BETA = 10.13    
    GAMMA = 1.5     # 假设 1/ln^3 的修正系数是 1.5
    
    total_points = 10000
    segment_size = 100
    tasks = []
    
    for i in range(0, total_points, segment_size):
        start_n = i
        end_n = min(i + segment_size, total_points)
        # 封装参数包
        tasks.append((i//segment_size, start_n, end_n, (K_BASE, BETA, GAMMA)))
    
    print(f"🚀 启动‘万点金矿’收割程序 | 并行度: 128")
    print(f">>> 参数配置: k0={K_BASE}, beta={BETA}, gamma={GAMMA}")
    
    t_start = time.time()
    
    # 针对 256 核，128 并发是带宽最舒适的区间
    with mp.Pool(processes=128) as pool:
        results = pool.map(harvest_worker, tasks)
    
    print("\n" + "="*60)
    for res in results:
        print(f"  {res}")
    print("="*60)
    print(f"✅ 万点收割任务完成！总耗时: {(time.time()-t_start)/60:.2f} 分钟")
    print(f">>> 数据存放在: {SAVE_DIR}，请执行后续的 Phase Unwrapping 统计误差谷底。")

✅ 成功加载权威真值，共 10000 个零点。
🚀 启动‘万点金矿’收割程序 | 并行度: 128
>>> 参数配置: k0=4.7, beta=10.13, gamma=1.5

  Seg 000 | n=1-100 | R2=0.965817 | AvgErr=2.316 | MaxErr=5.548 | 3213.8s
  Seg 001 | n=101-200 | R2=0.967871 | AvgErr=1.129 | MaxErr=2.918 | 3243.8s
  Seg 002 | n=201-300 | R2=0.956367 | AvgErr=1.284 | MaxErr=2.453 | 2921.4s
  Seg 003 | n=301-400 | R2=0.951789 | AvgErr=1.193 | MaxErr=2.481 | 3051.8s
  Seg 004 | n=401-500 | R2=0.948233 | AvgErr=1.220 | MaxErr=2.372 | 2780.3s
  Seg 005 | n=501-600 | R2=0.971149 | AvgErr=0.855 | MaxErr=1.787 | 2080.4s
  Seg 006 | n=601-700 | R2=0.974550 | AvgErr=0.793 | MaxErr=1.556 | 3400.7s
  Seg 007 | n=701-800 | R2=0.951428 | AvgErr=1.043 | MaxErr=2.558 | 3270.8s
  Seg 008 | n=801-900 | R2=0.972128 | AvgErr=0.788 | MaxErr=1.560 | 3369.8s
  Seg 009 | n=901-1000 | R2=0.975054 | AvgErr=0.747 | MaxErr=1.473 | 3087.5s
  Seg 010 | n=1001-1100 | R2=0.973512 | AvgErr=0.748 | MaxErr=1.747 | 3156.9s
  Seg 011 | n=1101-1200 | R2=0.951635 | AvgErr=0.995 | MaxErr=2.060 | 30